# SIGMOD Exp 4: Snapshot Concurrency

Snapshot-semantics concurrency benchmark over the maintained Q14 build-side state. `SNAP` rebuilds a retained snapshot from the heap-MVCC base and blocks readers that need a newer snapshot until that rebuild completes. `IVMH` keeps the latest derived hash current, but an old-snapshot reader still waits for on-demand snapshot materialization from the same heap base. MVHT variants serve old-snapshot reads directly from retained multi-version derived state and only make fresh reads wait for update commit. This notebook reports old/fresh blocking as numeric comparisons and uses a mixed old+fresh workload to visualize end-to-end completion time.

1. Fixed-point blocking summary (table)
2. Fixed-point mixed-workload total time
3. Update-volume sweep of mixed-workload total time

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_FIXED_UPDATE_PCT,
    SIGMOD_REPEAT,
    SIGMOD_TPCH_SF,
    SIGMOD_TRIM,
    SIGMOD_UPDATE_SWEEP_PCTS,
    SIGMOD_WARMUP,
    apply_paper_style,
    ensure_dirs,
    resolve_tpch_file,
    resolve_update_file,
    run_checked,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp4_concurrency').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

TPCH_DIR = (ROOT / 'benches' / 'sigmod' / 'tpch_data').resolve()
GEN_UPDATES = (ROOT / 'benches' / 'sigmod' / 'generate_updates.py').resolve()
BIN = ROOT / 'target' / 'release' / 'snapshot_concurrency_bench'

SF = SIGMOD_TPCH_SF
BUCKET_NUM = SIGMOD_BUCKET_NUM
WARMUP = SIGMOD_WARMUP
REPEAT = SIGMOD_REPEAT
TRIM = SIGMOD_TRIM
READ_TX_SIZE = 2048
FIXED_UPDATE_PCT = SIGMOD_FIXED_UPDATE_PCT
FIXED_READER_THREADS = 4
SWEEP_PCTS = list(SIGMOD_UPDATE_SWEEP_PCTS)
SERIES = [
    ('snap', 'nr'),
    ('ivmh', 'nr'),
    ('heap', 'wr'),
    ('chain', 'wr'),
    ('par', 'wr'),
]
STYLE = {
    ('snap', 'nr'): ('SNAP', TOL['red'], ':', 'x'),
    ('ivmh', 'nr'): ('IVMH', TOL['yellow'], '--', 'P'),
    ('heap', 'wr'): ('MONO-WR', TOL['blue'], '-', 'o'),
    ('chain', 'wr'): ('DUAL-WR', TOL['cyan'], '-', 's'),
    ('par', 'wr'): ('EPOCH-WR', TOL['green'], '-', 'D'),
}


def normalize_result_df(df):
    df = df.copy()
    for col in ('table_type', 'repair_mode'):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
    return df


PART_FILE = resolve_tpch_file(TPCH_DIR, 'part', SF)
LINEITEM_FILE = resolve_tpch_file(TPCH_DIR, 'lineitem_probe', SF, '_1995-09-01_1995-10-01.tbl')

print('PART    :', PART_FILE)
print('LINEITEM:', LINEITEM_FILE)
print('BIN     :', BIN)

In [ ]:
print('Building snapshot_concurrency_bench...')
run_checked(['cargo', 'build', '--release', '--bin', 'snapshot_concurrency_bench'], ROOT)
print('Build OK')

In [ ]:
def ensure_update_file(pct):
    try:
        return resolve_update_file(TPCH_DIR, SF, pct, 'uniform')
    except FileNotFoundError:
        print(f'Generating update file for {pct}%...')
        run_checked([sys.executable, str(GEN_UPDATES), str(PART_FILE), str(pct), str(SF), '--output-dir', str(TPCH_DIR)], ROOT)
        return resolve_update_file(TPCH_DIR, SF, pct, 'uniform')


def run_point(table_type, repair_mode, reader_threads, update_pct, output_csv):
    updates_file = ensure_update_file(update_pct)
    result = run_checked([
        str(BIN),
        '--part-file', str(PART_FILE),
        '--lineitem-file', str(LINEITEM_FILE),
        '--updates-file', str(updates_file),
        '--table-type', table_type,
        '--repair-mode', repair_mode,
        '--bucket-num', str(BUCKET_NUM),
        '--reader-threads', str(reader_threads),
        '--read-tx-size', str(READ_TX_SIZE),
        '--warmup', str(WARMUP),
        '--repeat', str(REPEAT),
        '--trim', str(TRIM),
        '--update-pct', str(update_pct),
        '--output-csv', str(output_csv),
    ], ROOT, quiet=True)
    for line in result.stderr.splitlines():
        if '[iter' in line or 'Average' in line or 'history_' in line or 'fresh_' in line or 'mixed_total_ms' in line:
            print(' ', line)

FIXED_CSV = DATA_DIR / 'sigmod_exp4_fixed.csv'
if FIXED_CSV.exists():
    FIXED_CSV.unlink()
for table_type, repair_mode in SERIES:
    print(f'fixed-point: {table_type}/{repair_mode}')
    run_point(table_type, repair_mode, FIXED_READER_THREADS, FIXED_UPDATE_PCT, FIXED_CSV)

df_fixed = normalize_result_df(pd.read_csv(FIXED_CSV))
display(df_fixed[[
    'table_type', 'repair_mode',
    'history_avg_read_wait_ms', 'history_avg_read_latency_ms',
    'fresh_avg_read_wait_ms', 'fresh_avg_read_latency_ms',
    'mixed_total_ms',
]])

In [ ]:
SWEEP_CSV = DATA_DIR / 'sigmod_exp4_update_sweep.csv'
if SWEEP_CSV.exists():
    SWEEP_CSV.unlink()
for pct in SWEEP_PCTS:
    for table_type, repair_mode in SERIES:
        print(f'update sweep: pct={pct} {table_type}/{repair_mode}')
        run_point(table_type, repair_mode, FIXED_READER_THREADS, pct, SWEEP_CSV)

df_sweep = normalize_result_df(pd.read_csv(SWEEP_CSV))
display(df_sweep[['table_type', 'repair_mode', 'update_pct', 'mixed_total_ms']].head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.0))

labels = []
mixed_values = []
mixed_colors = []
for key, (label, color, linestyle, marker) in STYLE.items():
    table_type, repair_mode = key
    sub = df_fixed[(df_fixed['table_type'] == table_type) & (df_fixed['repair_mode'] == repair_mode)]
    if sub.empty:
        raise ValueError(f'Missing fixed-point row for {table_type}/{repair_mode}')
    row = sub.iloc[0]
    labels.append(label)
    mixed_values.append(float(row['mixed_total_ms']))
    mixed_colors.append(color)

axes[0].bar(labels, mixed_values, color=mixed_colors, edgecolor='black', linewidth=0.4)
axes[0].set_title('Mixed Old+Fresh Workload')
axes[0].set_ylabel('Total Completion Time (ms)')
axes[0].tick_params(axis='x', rotation=25)
axes[0].grid(True, axis='y', linestyle='--', linewidth=0.6, alpha=0.6)

for key, (label, color, linestyle, marker) in STYLE.items():
    table_type, repair_mode = key
    sub = df_sweep[(df_sweep['table_type'] == table_type) & (df_sweep['repair_mode'] == repair_mode)].sort_values('update_pct')
    if sub.empty:
        raise ValueError(f'Missing sweep rows for {table_type}/{repair_mode}')
    axes[1].plot(sub['update_pct'], sub['mixed_total_ms'], color=color, linestyle=linestyle, marker=marker, linewidth=1.8, markersize=5, label=label)
axes[1].set_title('Mixed Workload vs Update Volume')
axes[1].set_xlabel('Update Volume (%)')
axes[1].set_ylabel('Total Completion Time (ms)')
axes[1].grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

handles, legend_labels = axes[1].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.10), framealpha=0.95)
fig.tight_layout()
out_pdf = FIGS_DIR / 'sigmod_exp4_concurrency.pdf'
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)